# Value Strategy — B/M ajusté intangibles · Qualité · Momentum · Détection ML de régimes

Notebook **narratif** : il importe la logique depuis le package `value_strategy` (dans `src/`) 
et raconte la stratégie partie par partie. Tout le code lourd vit dans les modules ; ce notebook 
ne fait qu'orchestrer et commenter.

**Stratégie** : long/short equity US Small/Mid Cap. Signal value = B/M ajusté intangibles 
(KC + OC, Peters & Taylor 2017), ranké par secteur, filtré qualité + momentum 6M. Short = bottom 
qualité du bucket growth. Détection ML des régimes *junk rally* pour réduire dynamiquement le short.

> **Données** : CRSP + Compustat via WRDS · **IS** 2003-2013 · **OOS** 2014-2024  
> Première exécution : connexion WRDS (met les données en cache). Ensuite : `USE_CACHE = True`.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# Rend le package importable depuis le notebook (src/ sur le path)
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from value_strategy import config, signals, factors, dynamic_short, plots, reports
from value_strategy import portfolio as pf
from value_strategy.data import build as data_build
from value_strategy.ml_regime import pipeline as ml_pipeline

config.ensure_dirs()
USE_CACHE = False   # passe à True pour relancer sans WRDS depuis le cache parquet

## Partie 1 — Préparation des données

Connexion WRDS, extraction CRSP + Compustat, correction du survivorship bias (Shumway 2001), 
calcul des capitaux immatériels KC/OC, nettoyage, illiquidité d'Amihud et coûts dynamiques. 
Le panel mensuel final est mis en cache (parquet) pour la reproductibilité.

In [ ]:
if USE_CACHE:
    panel, df_comp, crsp_ml, macro = data_build.load_cached_panel()
else:
    panel, df_comp, crsp_ml, macro = data_build.build_panel_from_wrds()
panel.shape

## Partie 2 — In-sample : construction et calibration (2003-2013)

Signaux value (neutralisés par secteur) + score qualité + momentum, puis construction des 
portefeuilles long/short (rebalancement semestriel) et performance nette de tous les coûts.

In [ ]:
panel_sm_is, filtered_is = signals.build_signals(panel, config.IS_START, config.IS_END)
long_is, short_is, _ = pf.construct_portfolios(filtered_is, config.IS_START, config.IS_END)
perf_is, costs_is = pf.compute_performance(panel_sm_is, long_is, short_is, config.IS_START, config.IS_END)

## Partie 3 — Out-of-sample : application et comparaison (2014-2024)

**Mêmes paramètres, aucun recalibrage.** On télécharge les facteurs Fama-French et on compare 
IS vs OOS (Sharpe, alpha FF4, information ratio vs HML).

In [ ]:
panel_sm_oos, filtered_oos = signals.build_signals(panel, config.OOS_START, config.OOS_END)
long_oos, short_oos, _ = pf.construct_portfolios(filtered_oos, config.OOS_START, config.OOS_END)
perf_oos, costs_oos = pf.compute_performance(panel_sm_oos, long_oos, short_oos, config.OOS_START, config.OOS_END)

ff5, mom_ff = factors.load_ff_factors()
stats_is, stats_oos = reports.print_is_oos_comparison(perf_is, perf_oos, ff5, mom_ff, costs_is, costs_oos)

## Partie 4 — Détection ML de régime « junk rally »

Feature engineering (CRSP + Compustat + FRED) → labeling HMM/GMM des régimes d'euphorie 
(VIX bas) → prédiction supervisée walk-forward → signaux `SHORT_REDUCE` / `FULL_SHORT`.

In [ ]:
ml = ml_pipeline.run_ml_regime(crsp_ml, df_comp, macro)
ml['metrics_df']

## Partie 5 — Réduction dynamique du short en junk rally

Quand le ML détecte l'euphorie, le poids du short passe de 100 % à 50 %. On compare la 
stratégie originale et la version *short dynamique* (Sharpe, MaxDD, alpha FF4, coûts).

In [ ]:
ff = factors.download_ff_factors()
perf_h, summary = dynamic_short.build_dynamic_short(perf_oos, ml['signals_df'], costs_oos)
perf_h, metrics = dynamic_short.attach_ff_and_metrics(perf_h, ff)
ff4_results = dynamic_short.ff4_regression(perf_h, ff)
reports.print_dynamic_short_verdict(perf_h, metrics, ff4_results, summary, costs_oos)

## Figures

Toutes les figures sont sauvegardées dans `results/charts/`.

In [ ]:
plots.setup_style()
plots.plot_cumulative_wealth(perf_is, perf_oos, ff5)
plots.plot_rolling_sharpe(perf_is, perf_oos, ff5)
plots.plot_annual_returns(perf_is, perf_oos, ff5)
plots.plot_drawdown(perf_is, perf_oos, ff5)
plots.plot_calendar_heatmap(perf_oos)
plots.plot_robustness(perf_is, perf_oos, ff5)
plots.plot_ml_dashboard(ml['mkt_df'], ml['results_df'], ml['importance_df'], ml['metrics_df'], ml['regime_method'], ml['avg_threshold'])
plots.plot_dynamic_short(perf_h, metrics, ff4_results, summary)
plots.plot_final_comparison(perf_is, perf_oos, perf_h, ff5, stats_is, stats_oos, metrics['adj'])